# [GIS] SGIS 2024 통계와 SK 월별 집계구 마스터 병합 — 2026-10-08

위에서부터 순서대로 실행하세요. GitHub 공유를 위해 프로젝트 기준 상대경로를 사용합니다. 데이터 배치는 [실행 안내](../docs/sgis-census-merge.md)를 참고하세요.

- SGIS의 `연도 + 집계구 코드 + 변수 코드` 중복을 검사하고, 변수 코드를 열로 펼칩니다.
- **2024년 SGIS를 고정된 지역 특성으로 사용**하여 2025년 각 월에 붙입니다.
- 마스터의 `TOT_OA_CD` 기준 왼쪽 결합이며 기존 행 순서와 셀 내용을 보존합니다.
- 새 열 이름은 `SGIS2024_to_in_001` 형식입니다. 변수 설명표에 한글 항목명을 저장합니다.
- 원본 `N/A`와 원본 행이 없는 경우 모두 통계값은 결측으로 남기되, 별도 상태표에서 구분합니다. 0으로 채우지 않습니다.
- 원본은 수정하지 않으며 실행마다 새 결과 폴더를 만듭니다.

기존 파일 확인 결과는 집계구 1,698개, 변수 297개, 마스터 10,188행 × 194열입니다.
코드는 이 숫자를 강제로 맞추지 않고 실제 파일에서 다시 계산합니다.
코드 일치는 공간 경계 동일성을 보장하지 않습니다. 마스터의 `BASE_DATE=20250630`과
SGIS 통계 경계 기준의 호환 여부는 별도 확인 대상입니다.

실행 환경: Python 3.11 이상, `uv sync --extra dev`로 준비한 프로젝트 환경.
난수 시드: 해당 없음(확률적 처리 없음). 주요 파라미터: SGIS_YEAR=2024, MASTER_YEAR=2025.
입력 마스터: `sk_통신인구_전체_집계구.csv`(2025년 7~12월), SGIS: 2024년 CSV 38개.


## 1. 준비
저장소 루트에서 `uv sync --extra dev`를 실행하고 프로젝트 `.venv` 커널을 선택하세요.
`xlrd`는 프로젝트 의존성에 포함되며 제공용 `.xls` 코드표를 읽습니다.


In [ ]:
# 의존성 설치: 저장소 루트에서 uv sync --extra dev

from pathlib import Path
from datetime import datetime
import json
import unicodedata
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 12)


## 2. 프로젝트 경로와 통계연도
현재 작업 폴더부터 상위 폴더를 탐색하여 `pyproject.toml`과 `src/`가 있는 저장소 루트를 찾습니다.
루트 또는 `notebooks/`에서 실행할 수 있으며 개인 컴퓨터의 절대경로는 사용하지 않습니다.

- 마스터: `data/processed/sk_통합_집계구/sk_통신인구_전체_집계구.csv`
- 코드표: `data/raw/reference_data/ref_code/3. 제공용 코드(statistics_code).xls`
- SGIS: `data/raw/reference_data/sgis/2024/{population,household,housing,business}/`
- 결과: `data/processed/sgis_merge/실행시각/`

SGIS 원본 CSV를 해당 분야 폴더에 배치하세요. 기존 집계구 집계 노트북의 마스터 출력 위치를 사용합니다.
다른 위치에 자료를 두었다면 아래 상대경로를 수정하세요. 입력과 결과는 기존 `.gitignore`에서 제외됩니다.


In [ ]:
SGIS_YEAR = '2024'
MASTER_YEAR = '2025'

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError(
        '프로젝트 루트를 찾지 못했습니다. pyproject.toml과 src가 있는 저장소에서 '
        '작업 폴더를 저장소 루트 또는 그 하위 폴더로 설정하세요.'
    )

PROJECT_ROOT = find_project_root()
MASTER_PATH = PROJECT_ROOT / 'data/processed/sk_통합_집계구/sk_통신인구_전체_집계구.csv'
CODEBOOK_PATH = PROJECT_ROOT / 'data/raw/reference_data/ref_code/3. 제공용 코드(statistics_code).xls'
SGIS_DIRS = {
    '인구': PROJECT_ROOT / 'data/raw/reference_data/sgis' / SGIS_YEAR / 'population',
    '가구': PROJECT_ROOT / 'data/raw/reference_data/sgis' / SGIS_YEAR / 'household',
    '주택': PROJECT_ROOT / 'data/raw/reference_data/sgis' / SGIS_YEAR / 'housing',
    '사업체·종사자': PROJECT_ROOT / 'data/raw/reference_data/sgis' / SGIS_YEAR / 'business',
}
OUTPUT_ROOT = PROJECT_ROOT / 'data/processed/sgis_merge'

def resolve_korean_path(path):
    """같아 보이지만 유니코드 조합 방식이 다른 한글 경로도 찾습니다."""
    path = Path(path).absolute()
    if path.exists():
        return path
    current = Path(path.anchor)
    for part in path.parts[1:]:
        if not current.is_dir():
            raise FileNotFoundError(f'데이터 폴더가 없습니다: {current}')
        matches = [p for p in current.iterdir()
                   if unicodedata.normalize('NFC', p.name)
                   == unicodedata.normalize('NFC', part)]
        if len(matches) != 1:
            raise FileNotFoundError(f'경로를 찾을 수 없거나 모호합니다: {path}')
        current = matches[0]
    return current

def relative_path(path):
    """보고서와 화면에 개인 컴퓨터의 절대경로를 남기지 않습니다."""
    return unicodedata.normalize('NFC', Path(path).relative_to(PROJECT_ROOT).as_posix())

MASTER_PATH = resolve_korean_path(MASTER_PATH)
CODEBOOK_PATH = resolve_korean_path(CODEBOOK_PATH)
SGIS_DIRS = {name: resolve_korean_path(p) for name, p in SGIS_DIRS.items()}
print('마스터:', relative_path(MASTER_PATH))
print('결과 저장 위치:', relative_path(OUTPUT_ROOT))


## 3. 원본 읽기와 중복 검사
마스터는 기존 셀 표기를 유지하도록 문자열로 읽습니다. 새로 추가할 SGIS 값은 숫자로 변환합니다.
SGIS CSV에는 헤더가 없으므로 첫 번째 데이터 행부터 모두 읽습니다.


In [ ]:
master = pd.read_csv(MASTER_PATH, dtype=str, encoding='utf-8-sig', keep_default_na=False)
required = {'TOT_OA_CD', 'STD_YM'}
if not required.issubset(master.columns):
    raise ValueError(f'마스터 필수 열 누락: {required - set(master.columns)}')
assert master['TOT_OA_CD'].str.fullmatch(r'[0-9]{14}').all(), '마스터 코드 형식 오류'
assert master['STD_YM'].str.fullmatch(r'[0-9]{6}').all(), '마스터 기준월 형식 오류'
assert master['STD_YM'].str[:4].eq(MASTER_YEAR).all(), '설정한 마스터 연도와 다릅니다.'
if master.duplicated(['STD_YM', 'TOT_OA_CD']).any():
    raise ValueError('마스터에 기준월 + 집계구 코드 중복이 있습니다.')

parts, manifest = [], []
for category, directory in SGIS_DIRS.items():
    paths = sorted(directory.glob('*.csv'))
    if not paths:
        raise FileNotFoundError(f'CSV가 없습니다: {directory}')
    for path in paths:
        raw = pd.read_csv(path, header=None, dtype=str,
                          encoding='utf-8-sig', keep_default_na=False)
        if raw.shape[1] != 4 or raw.empty:
            raise ValueError(f'4열 데이터 형식이 아닙니다: {path}')
        raw.columns = ['SGIS_YEAR', 'TOT_OA_CD', 'VARIABLE_CODE', 'VALUE_RAW']
        raw['CATEGORY'] = category
        raw['SOURCE_FILE'] = unicodedata.normalize('NFC', path.name)
        parts.append(raw)
        manifest.append({'category': category, 'path': relative_path(path), 'rows': len(raw)})

sgis = pd.concat(parts, ignore_index=True)
assert sgis['SGIS_YEAR'].eq(SGIS_YEAR).all(), 'SGIS에 다른 연도 자료가 섞여 있습니다.'
assert sgis['TOT_OA_CD'].str.fullmatch(r'[0-9]{14}').all(), 'SGIS 코드 형식 오류'
assert sgis['VARIABLE_CODE'].str.fullmatch(r'[A-Za-z0-9_]+').all(), '변수 코드 형식 오류'
keys = ['SGIS_YEAR', 'TOT_OA_CD', 'VARIABLE_CODE']
duplicates = sgis.duplicated(keys, keep=False)
if duplicates.any():
    display(sgis.loc[duplicates].head(20))
    raise ValueError('SGIS 키가 중복됩니다. 합계나 첫 값으로 처리하지 않고 중단합니다.')

is_source_na = sgis['VALUE_RAW'].eq('N/A')
sgis['VALUE'] = pd.to_numeric(sgis['VALUE_RAW'].mask(is_source_na), errors='coerce')
unexpected = sgis['VALUE'].isna() & ~is_source_na
if unexpected.any():
    display(sgis.loc[unexpected, ['SOURCE_FILE', 'VARIABLE_CODE', 'VALUE_RAW']].head(20))
    raise ValueError('N/A 외의 숫자가 아닌 값이 있습니다. 원본 의미를 확인하세요.')
sgis['VALUE_STATUS'] = 'numeric'
sgis.loc[is_source_na, 'VALUE_STATUS'] = 'source_na'

print(f'마스터: {master.shape[0]:,}행 × {master.shape[1]:,}열')
print(f'SGIS: {len(manifest)}개 파일, {len(sgis):,}행, 변수 {sgis.VARIABLE_CODE.nunique()}개')
display(sgis.groupby('CATEGORY').agg(원본행수=('VALUE_RAW', 'size'),
                                    변수수=('VARIABLE_CODE', 'nunique')))


## 4. 코드표 연결
`집계구·행정동` 시트에서 실제 CSV에 등장하는 코드만 가져옵니다.
산업분류는 **11차**를 선택합니다. 동일 코드가 8·9·10차에도 재사용되므로 코드만으로 이름을 고르면 안 됩니다.
건축년도는 최신 구간을 사용합니다. 해당 구간 제목은 `2015년~2023년`이나 목록에는
`ho_yr_020 = 2024년`까지 있으므로, 코드표 원문을 보존하고 확인 메모를 남깁니다.
변수 설명표에는 원본 파일명과 마스터 집계구 기준 결측 현황도 포함합니다.


In [ ]:
codebook = pd.read_excel(CODEBOOK_PATH, sheet_name='집계구·행정동',
                         header=None, dtype=str, engine='xlrd').fillna('')
variables = sorted(sgis['VARIABLE_CODE'].unique())
codebook['CODEBOOK_SECTION'] = codebook[2].replace('', pd.NA).ffill().fillna('')
industrial = codebook[4].str.startswith(('cp_', 'cp2_'))
housing_year = codebook[4].str.startswith('ho_yr_')
appropriate = (~industrial | codebook['CODEBOOK_SECTION'].str.contains('11차', regex=False))
appropriate &= (~housing_year | codebook['CODEBOOK_SECTION'].str.contains('2015년~2023년', regex=False))
dictionary = codebook.loc[codebook[4].isin(variables) & appropriate,
                          [3, 4, 'CODEBOOK_SECTION']].copy()
dictionary.columns = ['VARIABLE_NAME', 'VARIABLE_CODE', 'CODEBOOK_SECTION']
dictionary['MAPPING_NOTE'] = ''
dictionary.loc[dictionary['VARIABLE_CODE'].str.startswith('ho_yr_'), 'MAPPING_NOTE'] = (
    '코드표 최신 건축년도 구간 사용. 구간 제목은 2015년~2023년이나 2024년 항목까지 기재되어 있어 연도 표기 확인 필요.'
)
dictionary = dictionary.drop_duplicates()
if dictionary['VARIABLE_CODE'].duplicated().any():
    raise ValueError('코드표에 같은 코드의 서로 다른 이름이 있습니다.')
unmapped = sorted(set(variables) - set(dictionary['VARIABLE_CODE']))
if unmapped:
    raise ValueError(f'코드표에 없는 변수: {unmapped}')

metadata = sgis.groupby('VARIABLE_CODE').agg(
    CATEGORY=('CATEGORY', lambda s: ' / '.join(sorted(set(s)))),
    SOURCE_FILES=('SOURCE_FILE', lambda s: ' | '.join(sorted(set(s)))),
).reset_index()
dictionary = dictionary.merge(metadata, on='VARIABLE_CODE', validate='one_to_one')
dictionary['OUTPUT_COLUMN'] = 'SGIS' + SGIS_YEAR + '_' + dictionary['VARIABLE_CODE']
dictionary['SGIS_YEAR'] = SGIS_YEAR
dictionary = dictionary.sort_values('VARIABLE_CODE').reset_index(drop=True)
display(dictionary[['OUTPUT_COLUMN', 'VARIABLE_NAME', 'CATEGORY']].head(12))


## 5. 집계구 코드 대조
마스터에만 있는 코드는 병합 후 SGIS 값이 결측이 됩니다. SGIS에만 있는 코드는 마스터에 새 행으로 추가하지 않습니다.
현재 제공 자료는 두 방향 차집합 모두 0개입니다. 불일치가 생기면 아래 결과에서 확인할 수 있습니다.


In [ ]:
master_codes = set(master['TOT_OA_CD'])
sgis_codes = set(sgis['TOT_OA_CD'])
master_only = sorted(master_codes - sgis_codes)
sgis_only = sorted(sgis_codes - master_codes)

print(f'마스터 집계구: {len(master_codes):,}개')
print(f'SGIS 집계구: {len(sgis_codes):,}개')
print(f'일치: {len(master_codes & sgis_codes):,}개')
print(f'마스터에만 존재: {len(master_only):,}개 / SGIS에만 존재: {len(sgis_only):,}개')
if master_only or sgis_only:
    print('마스터에만 있는 코드:', master_only[:20])
    print('SGIS에만 있는 코드:', sgis_only[:20])

match_rows = []
for month, group in master.groupby('STD_YM', sort=True):
    codes = set(group['TOT_OA_CD'])
    match_rows.append({'STD_YM': month, 'MASTER_CODES': len(codes),
                       'MATCHED_CODES': len(codes & sgis_codes),
                       'UNMATCHED_CODES': len(codes - sgis_codes)})
monthly_match = pd.DataFrame(match_rows)
display(monthly_match)


## 6. 피벗과 병합
`pivot_table(..., aggfunc='sum')` 대신 중복을 허용하지 않는 `pivot`을 사용합니다.
`validate='many_to_one'`으로 마스터 행이 불어나는 잘못된 결합을 막습니다.
SGIS는 2024년으로 고정했으므로 병합 키는 집계구 코드 하나입니다.

상태표는 집계구당 한 행이며 값은 `numeric`(숫자), `source_na`(원본 N/A),
`no_record`(원본에 해당 집계구·변수 행 없음)입니다.


In [ ]:
prefix = f'SGIS{SGIS_YEAR}_'
wide = sgis.pivot(index='TOT_OA_CD', columns='VARIABLE_CODE', values='VALUE')
wide = wide.reindex(columns=variables).add_prefix(prefix)
wide.columns.name = None
collisions = set(master.columns) & set(wide.columns)
if collisions:
    raise ValueError(f'이미 같은 SGIS 열이 있습니다: {sorted(collisions)[:10]}')

merged = master.merge(wide.reset_index(), on='TOT_OA_CD', how='left',
                      sort=False, validate='many_to_one')
assert len(merged) == len(master), '병합 후 행 수가 달라졌습니다.'
pd.testing.assert_frame_equal(merged[master.columns], master)
assert merged.shape[1] == master.shape[1] + len(variables)

status_wide = sgis.pivot(index='TOT_OA_CD', columns='VARIABLE_CODE', values='VALUE_STATUS')
status_wide = status_wide.reindex(index=sorted(master_codes), columns=variables).fillna('no_record')
status_wide.columns.name = None
coverage = pd.DataFrame({
    'VARIABLE_CODE': variables,
    'NUMERIC_COUNT': status_wide.eq('numeric').sum().reindex(variables).to_numpy(),
    'SOURCE_NA_COUNT': status_wide.eq('source_na').sum().reindex(variables).to_numpy(),
    'NO_RECORD_COUNT': status_wide.eq('no_record').sum().reindex(variables).to_numpy(),
})
coverage['MISSING_RATE'] = (coverage['SOURCE_NA_COUNT'] + coverage['NO_RECORD_COUNT']) / len(master_codes)
dictionary = dictionary.merge(coverage, on='VARIABLE_CODE', validate='one_to_one')

# 결측 상태와 실제 숫자 여부가 모든 셀에서 일치하는지 확인
aligned_values = wide.reindex(sorted(master_codes))
assert (aligned_values.notna().to_numpy() == status_wide.eq('numeric').to_numpy()).all()
assert merged[list(wide.columns)].notna().to_numpy().sum() == (
    status_wide.eq('numeric').sum(axis=1).reindex(master['TOT_OA_CD']).sum()
)

print(f'병합 완료: {merged.shape[0]:,}행 × {merged.shape[1]:,}열')
print(f'추가 변수: {len(variables)}개 / 기존 행 순서와 셀 내용 보존 확인')
preview_columns = ['STD_YM', 'TOT_OA_CD'] + [prefix + code for code in
    ['to_in_001', 'to_ga_001', 'to_ho_001', 'to_fa_010', 'to_em_020'] if prefix + code in merged]
display(merged[preview_columns].head())


## 7. 새 파일로 저장

1. `master_sgis2024.csv`: 기존 마스터 + SGIS 변수 전체. 결측은 빈 칸으로 저장합니다.
2. `sgis2024_variable_dictionary.csv`: 코드·한글명·출처·변수별 결측 현황.
3. `sgis2024_value_status.csv`: 집계구별 결측 상태. 월별로 반복 저장하지 않습니다.
4. `merge_report.json`: 사용 경로, 연도, 매칭 결과, 원본 행 수, 검증 결과.

CSV는 UTF-8 BOM으로 저장합니다. 분석 시 코드는 문자열로 다시 읽으세요.
SGIS 연간 값은 6개월에 반복되므로 **월을 합칠 때 SGIS 인구·가구수를 더하지 않습니다.**


In [ ]:
OUTPUT_DIR = OUTPUT_ROOT / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

merged_path = OUTPUT_DIR / f'master_sgis{SGIS_YEAR}.csv'
dictionary_path = OUTPUT_DIR / f'sgis{SGIS_YEAR}_variable_dictionary.csv'
status_path = OUTPUT_DIR / f'sgis{SGIS_YEAR}_value_status.csv'
report_path = OUTPUT_DIR / 'merge_report.json'

merged.to_csv(merged_path, index=False, encoding='utf-8-sig', na_rep='')
dictionary.to_csv(dictionary_path, index=False, encoding='utf-8-sig')
status_wide.add_prefix(prefix).reset_index().to_csv(status_path, index=False, encoding='utf-8-sig')

# 저장된 파일에서도 기존 열의 모든 셀과 행 순서가 보존되었는지 확인
saved_original = pd.read_csv(merged_path, usecols=list(master.columns), dtype=str,
                             encoding='utf-8-sig', keep_default_na=False)
pd.testing.assert_frame_equal(saved_original[master.columns], master)

report = {
    'sgis_year': SGIS_YEAR,
    'master_year': MASTER_YEAR,
    'master_path': relative_path(MASTER_PATH),
    'codebook_path': relative_path(CODEBOOK_PATH),
    'source_files': manifest,
    'master_rows': len(master), 'master_columns': len(master.columns),
    'sgis_rows': len(sgis), 'sgis_variables': len(variables),
    'merged_rows': len(merged), 'merged_columns': len(merged.columns),
    'master_unique_codes': len(master_codes), 'sgis_unique_codes': len(sgis_codes),
    'matched_unique_codes': len(master_codes & sgis_codes),
    'master_only_codes': master_only, 'sgis_only_codes': sgis_only,
    'monthly_match': monthly_match.to_dict(orient='records'),
    'original_na_rows': int(is_source_na.sum()),
    'master_base_dates': sorted(master['BASE_DATE'].unique()) if 'BASE_DATE' in master else [],
    'master_cells_preserved_after_save': True,
    'geographic_boundary_equality_verified': False,
    'missing_status_definitions': {'numeric': '원본 숫자',
                                   'source_na': '원본 N/A', 'no_record': '해당 원본 행 없음'},
    'year_policy': '2024 SGIS 고정 특성을 2025년 모든 월에 연결',
}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
for path in [merged_path, dictionary_path, status_path, report_path]:
    print(relative_path(path))
print('저장 후 기존 마스터 셀 보존 검증 완료')


## 8. 결과 다시 읽기
저장 후 다른 분석에서 사용하는 예시입니다. 아래 셀은 선택적으로 실행하세요.


In [ ]:
# result = pd.read_csv(
#     merged_path, encoding='utf-8-sig',
#     dtype={'TOT_OA_CD': str, 'STD_YM': str, 'ADM_CD': str, 'BASE_DATE': str},
# )
# result.head()
